# 配置训练任务

上一节我们了解了本章三步闭环的目标。这一节回答三个核心问题：数据管线、训练配置、并行策略。

---

## 1. Wordle 数据概览

我们用的数据集叫 `willcb/V3-wordle`。每条样本记录了一局完整的 Wordle 游戏：

```json
{
  "prompt": [
    {"role": "system", "content": "You are a Wordle playing assistant..."},
    {"role": "user", "content": "Round 1 feedback: X-X-X-X-X"},
    {"role": "assistant", "content": "<guess>CRANE</guess>"},
    {"role": "user", "content": "Round 2 feedback: G-X-X-X-Y"},
    {"role": "assistant", "content": "<guess>STARE</guess>"},
    ...
  ],
  "completion": [{"role": "assistant", "content": "<guess>SMILE</guess>"}]
}
```

| 字段 | 用途 |
|------|------|
| `prompt` | 多轮对话（6-10 条消息），**训练用** |
| `completion` | 最后一轮回复，**评测用**，训练不碰 |

---

## 2. 训练配置

打开 `torchtitan_npu/models/qwen3/config_registry.py`，`sft_qwen3_1_7b_wordle()` 函数里写满了配置。对于初学者，关注四块：

```python
# ① 数据管线
dataloader=ChatDataLoader.Config(
    dataset_path="willcb/V3-wordle",
    load_dataset_kwargs={"split": "train"},
    sample_processor=_process_wordle_sample,
)

# ② 训练配置
training=TrainingConfig(
    local_batch_size=2,
    global_batch_size=64,
    seq_len=1024,
    max_norm=1.0,
    steps=20,
)

# ③ 学习率调度
lr_scheduler=LRSchedulersContainer.Config(
    warmup_steps=0,
    decay_ratio=1.0,
    decay_type="cosine",
    min_lr_factor=1.0,
)

# ④ 并行策略
parallelism=ParallelismConfig(
    data_parallel_replicate_degree=1,
    data_parallel_shard_degree=-1,
    tensor_parallel_degree=1,
    pipeline_parallel_degree=1,
    expert_parallel_degree=1,
    expert_tensor_parallel_degree=1,
    context_parallel_degree=1,
)
```

下面逐一展开。

---

## 3. 数据加载

配置里 `dataloader` 只写了三行，背后做了什么？简单说就是三步：

```text
原始 JSON → 渲染成文本 → 转成数字 → 标记哪些算 loss
```

### 3.1 Chat Template

模型输入是 token 序列，不是 JSON。第一步是把对话消息变成一段带特殊标记的文本。

Qwen3 用 `<|im_start|>` 和 `<|im_end|>` 标记每条消息的边界：

```text
<|im_start|>user
法国的首都是哪里？<|im_end|>
<|im_start|>assistant
巴黎。<|im_end|>
```

不同模型的标记不一样，但思路相同：用特殊 token 告诉模型「谁在说话」。

### 3.2 Tokenize

把文本送给 tokenizer，每个词变成对应的整数 ID，得到 `input_ids` 序列。

### 3.3 Label Mask

核心问题：SFT 的目的是让模型**学会怎么回复**，不是学会用户会说什么。所以：

- **user 和 system 说的内容** → 不学（label 设成 -100，PyTorch 会自动跳过）
- **assistant 说的内容** → 学（label 保留正确的 token ID）

![SFT Label Masking](./images/sft-label-masking.png)

> **直觉**：SFT 不是在整段对话上算 loss，而是只在 assistant 的回复上算。这就是 label mask 做的事情。

---

## 4. 训练运行

下面展开训练配置。

### 4.1 梯度累计

```python
training=TrainingConfig(
    local_batch_size=2,     # 每卡每次处理 2 条样本
    global_batch_size=64,    # 累计 64 条样本后更新参数
    ...
)
```

显存放不下太多样本，每次只能处理 2 条。但我们希望用 64 条样本来算一次更新（样本多，梯度估算更准）。

**梯度累计**：做 2 次前向，把梯度攒起来，攒够了再一次性更新。

```text
前向 #1: 吃 2 条 → 算梯度，攒着不更新
前向 #2: 吃 2 条 → 算梯度，攒够了 → 更新参数！
```

公式：`梯度累计次数 = global_batch_size(64) ÷ local_batch_size(2) = 32`


### 4.2 其他参数

| 参数 | 值 | 说明 |
|------|-----|------|
| `seq_len` | 1024 | 每条样本最多保留 1024 个 token，超了截断 |
| `max_norm` | 1.0 | 梯度裁剪：如果某步梯度太大，压到 1.0 以内，防止训练震荡 |
| `steps` | 20 | 总共训练 20 步。20 步足够学会 `<guess>` 格式，但不够学会策略猜词（那是 RL 阶段的事） |

### 4.3 学习率

```python
lr_scheduler=LRSchedulersContainer.Config(
    warmup_steps=0,        # 不需要热身
    decay_ratio=1.0,       # 全程衰减
    decay_type="cosine",   # 余弦曲线衰减
    min_lr_factor=1.0,     # 最小 lr = 初始 lr × 1.0 = 不变
)
```

学习率控制参数更新的步幅。`lr=1e-5` 初始值由 optimizer 配置给出。

这四个参数组合在一起意味着什么？`min_lr_factor=1.0` 表示衰减终点等于起点 → **实际上没有衰减**。cosine 曲线从 1e-5 "衰减"到 1e-5，等于一条水平线。

**为什么？** 只跑 20 步，学习率还没来得及衰减就结束了。如果是 2000 步的长训练，就要把 `min_lr_factor` 设成 0.1 或更小，让学习率慢慢降下来帮助收敛。

---

## 5. 多卡训练

以上假设单卡训练。但实际训练往往需要多张卡——模型太大，一张卡放不下，或者需要加速。

### 5.1 并行配置选项

框架提供了七种并行维度，对应七个配置参数：

```python
parallelism=ParallelismConfig(
    data_parallel_replicate_degree=1,    # DP 副本（DDP）
    data_parallel_shard_degree=-1,       # DP 分片（FSDP），-1=自动
    tensor_parallel_degree=1,            # TP 张量并行
    pipeline_parallel_degree=1,          # PP 流水线并行
    expert_parallel_degree=1,            # EP Expert并行
    expert_tensor_parallel_degree=1,     # ETP Expert+张量并行
    context_parallel_degree=1,           # CP 上下文并行
)
```

它们各自解决不同的问题：

| 并行策略 | 配置项 | 思路 | 适用场景 |
|----------|--------|------|-----------|
| **DP (FSDP)** | `data_parallel_shard_degree` | 数据分到多卡，参数也可切分省显存 | **首选**，既加速又省显存 |
| DP (DDP) | `data_parallel_replicate_degree` | 数据分到多卡，每卡存完整模型副本 | 模型小、卡多、纯加速 |
| **TP** | `tensor_parallel_degree` | 把一层矩阵乘法切开放到多卡 | 单层太大，一张卡放不下（>10B 模型） |
| **PP** | `pipeline_parallel_degree` | 不同层放到不同卡，像流水线 | 层数太多（>100 层） |
| **CP** | `context_parallel_degree` | 长序列切开分到多卡 | seq_len 超长（>128K） |
| **EP** | `expert_parallel_degree` | MoE 的 expert 分配到不同卡 | 模型有 MoE 结构（如 DeepSeek） |
| **ETP** | `expert_tensor_parallel_degree` | EP + TP 组合 | MoE 且 expert 也很大 |

### 5.2 配置选择

对 Qwen3-1.7B + Wordle SFT 这个任务：

| 条件 | 判断 | 结论 |
|------|------|------|
| 模型 1.7B，单层不大 | 一张卡能放下单层 | TP=1 |
| 共 28 层 | 不算多 | PP=1 |
| seq_len=1024 | 不算长 | CP=1 |
| 不是 MoE | 没有 expert | EP=1, ETP=1 |
| ✅ **多卡时既要加速又要省显存** | 第 2 章讲过 FSDP 能做到这两点 | **选 DP（FSDP）** |

结论：七个参数中，只需要用 `data_parallel_shard_degree`。其他全设 1。

`data_parallel_shard_degree=-1` 是自动模式：
- 1 张卡 → 自动=1，不分片（本章的情况）
- 2 张卡 → 自动=2，参数切成两份，每卡一半
- 8 张卡 → 自动=8，参数切成八份

> 本章用单卡（`NGPU=1`），FSDP 实际未启动。改成 `NGPU=2`，一样的配置就能用上多卡分片。

### 5.3 并行策略选择指南

```text
模型 < 7B、层数不多、序列不长  →  DP (FSDP) 就够
模型 > 10B，单层放不下         →  DP + TP
层数 > 100                     →  DP + PP
序列 > 128K                    →  DP + CP
是 MoE 模型                    →  DP + EP
```

---

## 6. Activation Checkpoint

此外，Activation Checkpoint 可以进一步节省显存：

```python
activation_checkpoint=ActivationCheckpointConfig(mode="selective")
```

前向传播时，每一层都产生中间结果（激活值），反向时需要用到。全存着显存撑爆。

AC 的做法：**只保留关键位置的激活值，其他丢掉。反向需要时临时重算。** 本质是「计算换显存」——多花 20-30% 时间，省 30-40% 显存。`selective` 模式只在部分层做，性价比最高。

---

## 小结

本节走完了从数据到配置的完整链路：

- **数据**：`sample_processor` 把对话 JSON 变成 `(input_ids, labels)`，关键是只在 assistant 回复上算 loss（label mask）。
- **训练**：`local_batch_size` × 梯度累计 = `global_batch_size` 决定多少样本更新一次。
- **学习率**：20 步太短，设成恒定（`min_lr_factor=1.0`）。
- **多卡**：DP（FSDP）既能加速又能省显存，1.7B 模型选它就够，TP/PP/CP/EP 留给更大的模型。
- **省显存**：activation checkpoint 用计算换空间。

下一节将使用这份配置启动训练。

## 练习

1. （单选题）`sample_processor` 在数据管线中扮演什么角色？
    A. 下载数据集
    B. 将原始 JSON 变成 `(input_ids, labels)` 对
    C. 控制 batch 大小
    D. 启动训练循环

2. （单选题）`local_batch_size=2, global_batch_size=4` 意味着什么？
    A. 每次前向用 4 个样本，每 2 步更新一次参数
    B. 每次前向用 2 个样本，每 2 次前向梯度累计后更新一次参数
    C. 数据集总共只有 4 个样本
    D. 模型有 4 层，每层 2 个 batch

3. （判断题）`min_lr_factor=1.0` 意味着学习率实际上不衰减，cosine 曲线退化为一条水平线。20 步训练太短，这是合理的。

4. （单选题）Qwen3-1.7B 训练时，为什么 `tensor_parallel_degree=1`？
    A. TP 已过时
    B. 模型只有 1.7B，单层不大，一张卡放得下，不需要切分单层
    C. 只有 NLP 模型才用 TP
    D. TP 需要 8 张卡以上

5. （单选题）`data_parallel_shard_degree=-1` 的含义是什么？
    A. 手动指定不分片
    B. 自动配置——框架根据卡数自动决定分片数
    C. 禁用 FSDP
    D. 固定为单卡模式

6. （判断题）1.7B 模型 + seq_len=1024 + 28 层这个规模，只需要 DP（FSDP），TP/PP/CP/EP 都留给更大的模型。

In [ ]:
!cat ./answer/03.02_answer.txt